[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/44_alibi_solution.ipynb)

# 🟡 Solution: ALiBi Attention

Reference solution for ALiBi (Attention with Linear Biases).

$$m_h = \frac{1}{2^{8h/H}}, \quad \text{bias} = -m_h \cdot |i - j|$$

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def alibi_attention(Q, K, V, num_heads):
    B, S, D = Q.shape
    d_h = D // num_heads

    h_idx = torch.arange(1, num_heads + 1, dtype=torch.float32, device=Q.device)
    slopes = 1.0 / (2.0 ** (8.0 * h_idx / num_heads))

    pos = torch.arange(S, device=Q.device).float()
    dist = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs()

    bias = -slopes.view(num_heads, 1, 1) * dist.unsqueeze(0)

    Qh = Q.view(B, S, num_heads, d_h).transpose(1, 2)
    Kh = K.view(B, S, num_heads, d_h).transpose(1, 2)
    Vh = V.view(B, S, num_heads, d_h).transpose(1, 2)

    scores = (Qh @ Kh.transpose(-2, -1)) / (d_h ** 0.5) + bias.unsqueeze(0)
    attn = torch.softmax(scores, dim=-1)
    out = (attn @ Vh).transpose(1, 2).reshape(B, S, D)
    return out

In [ ]:
# Verify
torch.manual_seed(0)
B, S, D, H = 2, 6, 16, 4
Q = torch.randn(B, S, D)
K = torch.randn(B, S, D)
V = torch.randn(B, S, D)

out = alibi_attention(Q, K, V, num_heads=H)
print("Output shape:", out.shape)

h_idx = torch.arange(1, H + 1, dtype=torch.float32)
slopes = 1.0 / (2.0 ** (8.0 * h_idx / H))
print("Slopes for 4 heads:", slopes)

In [ ]:
from torch_judge import check
check("alibi")